# Available data exploration

**Purpose.** Take stock of the datasets already downloaded for the thesis,
and identify which data are still missing to answer RQ1–RQ3 (see
`docs/research_questions.md`).

**Scope.** Exploratory only — no modelling, no filtering decisions. All data
here are *pilot downloads* for access validation (see `data/README.md`), not
a species/region decision. The per-dataset deep dives used to live in this
notebook; they are now split by folder, per `notebooks/README.md`, and are
the authoritative source for their own facts (not repeated here):

| Dataset | Location | Explored in |
|---|---|---|
| iNaturalist *Acacia*, Coimbra district | `data/raw/citizen_science/` | `../02_citizen_science/acacia_coimbra_pilot_exploration.ipynb` |
| TESSERA embeddings 2024, Coimbra municipality | `data/raw/satellite/tessera_coimbra_pilot/` | `../03_satellite/tessera_embeddings_coimbra_pilot.ipynb` |
| Linking the two (feasibility check) | — | `../04_data_integration/link_observations_to_embeddings_pilot.ipynb` |
| COS 2023 (national land cover) | `data/raw/satellite/cos/` | not explored yet |

This notebook keeps only what's dataset-agnostic: the file inventory below,
and the gap list at the end.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"

pd.set_option("display.max_columns", 50)

## 1. Inventory of downloaded files

In [ ]:
rows = []
for f in sorted(RAW.rglob("*")):
    if f.is_file() and f.name != ".gitkeep":
        rel = f.relative_to(RAW)
        rows.append({"group": "/".join(rel.parts[:2]), "file": f.name, "size_MB": f.stat().st_size / 1e6})
inventory = pd.DataFrame(rows)
inventory.groupby("group").agg(n_files=("file", "size"), size_MB=("size_MB", "sum")).round(1)

## 2. What data are still missing (vs. RQ1–RQ3)

For dataset-specific established facts and open questions, see the
notebooks linked above — `02_citizen_science/` and `03_satellite/` each
close with their own summary, and are the authoritative source for their
facts (not repeated here, to avoid the two notebooks drifting apart).

| Need | Why (RQ) | Current status | Label |
|---|---|---|---|
| **Conventional Sentinel-2 (and possibly Sentinel-1) features** over the study area | RQ1, RQ2 baseline — the comparison is impossible without it | Not obtained (needs Copernicus Data Space / GEE account) | Established fact (gap) |
| **Aligned study area** — TESSERA for the whole district, or iNat clipped to the municipality | All RQs; most observations currently have no embedding | Boundaries not reconciled | Established fact (gap) |
| **A second, geographically distinct region** (same or comparable species) | RQ3 — the supervisor-confirmed comparison is *across* geographically distinct areas; this single-region pilot cannot provide that by itself | Not obtained | Established fact (gap) |
| **A defined distribution-improving filter for RQ2** (criterion for which observations to remove, in a densely-observed area) | RQ2, per supervisor refinement 2026-10-17 | Not defined or implemented | Open question |
| **Negative / background samples** | Any classifier needs a negative class; presence-only data provide none | No strategy, no data | Open question |
| **Independent validation reference** (not from the same citizen-science source) | Evaluation of all RQs; RQ3 in particular, since biased training data evaluated on equally biased test data hides the bias effect | None | Open question |
| **Bias covariates** — roads/trails, accessibility (e.g. OpenStreetMap), population | RQ3, per Dimson et al. (2023) bias axes in `research_gaps.md` | Not obtained | Reasonable proposal |
| **Study-area boundary** (CAOP administrative limits) | Clipping, area statistics, mapping | Not obtained (currently using bounding boxes) | Reasonable proposal |
| **More citizen-science records** — other iNat years/regions, or GBIF (which also aggregates other sources) | RQ2 needs enough data for the filtering approach above; RQ3 needs a second region, not just spatial variety within one | Only one pilot query | Open question |
| **AlphaEarth (or another) foundation model** | RQ1 if more than one FM is compared | Not obtained, access unconfirmed | Open question |
| **COS 2023** | Land-cover context, possible negatives/validation | Downloaded; exploration deferred | — |